In [2]:
## 1. Configuração da sessão

In [5]:
%idle_timeout 30
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2


In [1]:
import time
import boto3
from pyspark.context import SparkContext
from awsglue.context import GlueContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

print("Sessão Spark iniciada.")

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 30
Session ID: 5a4c76a3-fd2d-41ed-8d9c-4246bf97cba1
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 5a4c76a3-fd2d-41ed-8d9c-4246bf97cba1 to get into ready status...
Session 5a4c76a3-fd2d-41ed-8d9c-4246bf97cba1 has been created.
Sessão Spark iniciada.


In [2]:
BUCKET = "teste-fiap-fase3"
PREFIXO_BRONZE = "bronze/"
DATABASE_BRONZE = "datahackers_bronze"
CRAWLER_BRONZE = "crawler_bronze"
CLASSIFICADOR = "csv_datahackers"
ROLE = "LabRole"

# Extrações por ano -> subpasta na Bronze
EDICOES = {
    "2023": "state_of_data_2023",
    "2024": "state_of_data_2024",
    "2025-2026": "state_of_data_2025_2026",
}

s3 = boto3.client("s3")
glue = boto3.client("glue")

In [3]:
# Confirmação de que cada extração está na pasta correta
for ano, pasta in EDICOES.items():
    resposta = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{PREFIXO_BRONZE}{pasta}/")
    arquivos = [o for o in resposta.get("Contents", []) if o["Key"].endswith(".csv")]
    if len(arquivos) != 1:
        raise ValueError(f"{ano}: esperava 1 CSV em {pasta}/, encontrei {len(arquivos)}.")
    arq = arquivos[0]
    print(f"{ano:<10} s3://{BUCKET}/{arq['Key']}  |  {arq['Size']/1024/1024:.1f} MB  |  "
          f"{arq['LastModified']:%d/%m/%Y %H:%M}")

2023       s3://teste-fiap-fase3/bronze/state_of_data_2023/State_of_data_BR_2023_Kaggle - df_survey_2023.csv  |  14.5 MB  |  22/10/2026 22:17
2024       s3://teste-fiap-fase3/bronze/state_of_data_2024/Final Dataset - State of Data 2024 - Kaggle - df_survey_2024.csv  |  15.5 MB  |  22/10/2026 22:18
2025-2026  s3://teste-fiap-fase3/bronze/state_of_data_2025_2026/Final Dataset - State of Data 2025-2026 - Kaggle.csv  |  9.9 MB  |  22/10/2026 22:19


In [4]:
# Leitura com Spark
def ler_bronze(pasta):
    return (
        spark.read
        .option("header", "true")
        .option("multiLine", "true")
        .option("quote", '"')
        .option("escape", '"')
        .option("encoding", "UTF-8")
        .option("inferSchema", "false")
        .csv(f"s3://{BUCKET}/{PREFIXO_BRONZE}{pasta}/")
    )

bronze = {}
for ano, pasta in EDICOES.items():
    df = ler_bronze(pasta)
    bronze[ano] = df
    total = df.count()
    duplicadas = total - df.dropDuplicates().count()
    print(f"{ano:<10} {total:>6,} registros | {len(df.columns)} colunas | duplicadas: {duplicadas}")
    print("           primeiras colunas:", df.columns[:3])

2023        5,293 registros | 399 colunas | duplicadas: 0
           primeiras colunas: ["('P0', 'id')", "('P1_a ', 'Idade')", "('P1_a_1 ', 'Faixa idade')"]
2024        5,217 registros | 403 colunas | duplicadas: 2
           primeiras colunas: ['0.a_token', '0.d_data/hora_envio', '1.a_idade']
2025-2026   3,495 registros | 388 colunas | duplicadas: 1
           primeiras colunas: ['0.a_token', '0.d_data/hora_envio', '1.a_idade']


In [5]:
# Confirmação dos arquivos em parquet e se as colunas foram corretamente noemadas
for ano, pasta in EDICOES.items():
    df_parquet = spark.read.parquet(f"s3://{BUCKET}/bronze/parquet/{pasta}/")
    df_csv = bronze[ano]
    status = "OK" if df_parquet.count() == df_csv.count() and len(df_parquet.columns) == len(df_csv.columns) else "DIVERGENTE"
    print(f"{ano:<10} CSV: {df_csv.count():>6,} reg / {len(df_csv.columns)} col | "
          f"Parquet: {df_parquet.count():>6,} reg / {len(df_parquet.columns)} col | {status}")

2023       CSV:  5,293 reg / 399 col | Parquet:  5,293 reg / 399 col | OK
2024       CSV:  5,217 reg / 403 col | Parquet:  5,217 reg / 403 col | OK
2025-2026  CSV:  3,495 reg / 388 col | Parquet:  3,495 reg / 388 col | OK
